# Axial v3 Iteration A - raw_0 audit

Diagnostico train/validation de `raw_0` sin interpretar la etiqueta como anatomia.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from lumbar_mri.axial_v3.audit import audit_raw0_slice, summarize_raw0_by_patient, write_audit_outputs
from lumbar_mri.axial_v3.calibration import apply_raw0_threshold, raw0_presence_metrics
from lumbar_mri.axial_v3.guards import require_train_val_only

ALLOWED_SPLITS = require_train_val_only(["train", "val"], context="notebook_51")
OUTPUT_ROOT = Path("outputs") / "axial_v3" / "iteration_a"
RAW0_CLASS_INDEX = 1

## Output schema

Planned outputs: `raw0_slice_audit.csv`, `raw0_patient_audit.csv`, `raw0_audit_summary.json`, figures, mosaics, and a human-review candidate list. No results are populated until the notebook is run on train/validation data.

In [ ]:
def audit_records(records, load_mask, load_image_shape):
    rows = []
    for record in records:
        require_train_val_only([record.split], context="notebook_51.audit_records")
        mask = load_mask(record)
        rows.append(
            audit_raw0_slice(
                mask=mask,
                split=record.split,
                patient_id=record.patient_id,
                study_id=record.study_id,
                slice_id=record.slice_id,
                image_shape=load_image_shape(record),
                raw0_value=RAW0_CLASS_INDEX,
                background_value=0,
            )
        )
    patient_summary = summarize_raw0_by_patient(rows)
    write_audit_outputs(rows, OUTPUT_ROOT)
    return rows, patient_summary


def validation_probability_probe(probabilities, target_mask, min_probability=0.5, min_margin=0.0):
    prediction = apply_raw0_threshold(
        probabilities,
        raw0_index=RAW0_CLASS_INDEX,
        min_probability=min_probability,
        min_margin=min_margin,
    )
    return raw0_presence_metrics(prediction, target_mask, raw0_index=RAW0_CLASS_INDEX)

## Human review pending

Representative cases should be selected after running the audit: positives, negatives, border-contact cases, smallest annotations, largest annotations, and probability-discordant validation cases.